# Inter-Agent Message Validation

Agent 간 메시지는 내부 통신처럼 보이지만 보안 관점에서는 신뢰할 수 없는 입력입니다.

이번 실습에서는 메시지 envelope를 표준화하고, 스키마 검증과 정책 검증을 통과한 메시지만 전달합니다.


In [ ]:
from pprint import pprint
from typing import Any, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field, ValidationError, field_validator


## 1단계: 메시지 Envelope 정의

모든 Agent 메시지는 같은 envelope를 사용합니다.


In [ ]:
AgentId = Literal["intake", "research", "finance", "security"]
Intent = Literal["handoff", "tool_request", "result", "escalation"]
Sensitivity = Literal["public", "internal", "restricted"]


class AgentMessage(BaseModel):
    trace_id: str = Field(min_length=8)
    sender: AgentId
    recipient: AgentId
    intent: Intent
    capability: str | None = None
    sensitivity: Sensitivity
    payload: dict[str, Any]

    @field_validator("payload")
    @classmethod
    def payload_must_have_summary(cls, value: dict[str, Any]) -> dict[str, Any]:
        if "summary" not in value:
            raise ValueError("payload.summary is required")
        return value


## 2단계: 송수신 경로와 Capability 정책

스키마가 맞아도 허용되지 않은 송수신 경로라면 차단해야 합니다.


In [ ]:
ALLOWED_EDGES = {
    ("intake", "research"),
    ("intake", "finance"),
    ("research", "intake"),
    ("finance", "intake"),
    ("intake", "security"),
    ("security", "intake"),
    ("security", "finance"),
}


RECIPIENT_CAPABILITIES = {
    "intake": {"case.route", "case.reply"},
    "research": {"public.docs.search", "public.docs.summarize"},
    "finance": {"finance.invoice.read", "finance.refund.draft"},
    "security": {"audit.log.read", "security.incident.create"},
}


RECIPIENT_SENSITIVITY = {
    "intake": {"public", "internal"},
    "research": {"public", "internal"},
    "finance": {"internal", "restricted"},
    "security": {"public", "internal", "restricted"},
}


## 3단계: LangGraph 상태와 검증 노드


In [ ]:
class MessageState(TypedDict, total=False):
    raw_message: dict
    validated_message: AgentMessage
    validation_errors: list[str]
    delivery_result: str
    audit_log: list[str]


def validate_message(state: MessageState) -> MessageState:
    audit = state.get("audit_log", [])
    errors: list[str] = []

    try:
        message = AgentMessage.model_validate(state["raw_message"])
    except ValidationError as error:
        return {
            **state,
            "validation_errors": [str(error)],
            "audit_log": audit + ["schema:denied"],
        }

    if (message.sender, message.recipient) not in ALLOWED_EDGES:
        errors.append(f"edge_not_allowed:{message.sender}->{message.recipient}")

    if message.capability is not None:
        allowed = RECIPIENT_CAPABILITIES[message.recipient]
        if message.capability not in allowed:
            errors.append(f"capability_not_allowed:{message.recipient}:{message.capability}")

    if message.sensitivity not in RECIPIENT_SENSITIVITY[message.recipient]:
        errors.append(f"sensitivity_not_allowed:{message.recipient}:{message.sensitivity}")

    if errors:
        return {
            **state,
            "validated_message": message,
            "validation_errors": errors,
            "audit_log": audit + [f"policy:denied:{errors}"],
        }

    return {
        **state,
        "validated_message": message,
        "validation_errors": [],
        "audit_log": audit + [f"policy:approved:{message.trace_id}"],
    }


def route_by_validation(state: MessageState) -> str:
    if state.get("validation_errors"):
        return "reject_message"
    return "deliver_message"


In [ ]:
def deliver_message(state: MessageState) -> MessageState:
    message = state["validated_message"]
    return {
        **state,
        "delivery_result": f"{message.sender} -> {message.recipient} 전달 완료",
        "audit_log": state.get("audit_log", []) + [f"deliver:{message.trace_id}"],
    }


def reject_message(state: MessageState) -> MessageState:
    return {
        **state,
        "delivery_result": "메시지 전달 차단",
        "audit_log": state.get("audit_log", []) + ["deliver:blocked"],
    }


In [ ]:
workflow = StateGraph(MessageState)

workflow.add_node("validate_message", validate_message)
workflow.add_node("deliver_message", deliver_message)
workflow.add_node("reject_message", reject_message)

workflow.add_edge(START, "validate_message")
workflow.add_conditional_edges(
    "validate_message",
    route_by_validation,
    {
        "deliver_message": "deliver_message",
        "reject_message": "reject_message",
    },
)
workflow.add_edge("deliver_message", END)
workflow.add_edge("reject_message", END)

graph = workflow.compile()


## 4단계: 메시지 검증 테스트


In [ ]:
valid_message = {
    "trace_id": "TRACE-001",
    "sender": "intake",
    "recipient": "research",
    "intent": "handoff",
    "capability": "public.docs.search",
    "sensitivity": "public",
    "payload": {"summary": "LangGraph 보안 자료 검색"},
}

invalid_edge = {
    "trace_id": "TRACE-002",
    "sender": "research",
    "recipient": "finance",
    "intent": "tool_request",
    "capability": "finance.invoice.read",
    "sensitivity": "internal",
    "payload": {"summary": "청구서 조회 요청"},
}

invalid_capability = {
    "trace_id": "TRACE-003",
    "sender": "intake",
    "recipient": "research",
    "intent": "tool_request",
    "capability": "finance.invoice.read",
    "sensitivity": "internal",
    "payload": {"summary": "연구 Agent에게 금융 조회를 위임"},
}

invalid_schema = {
    "trace_id": "BAD",
    "sender": "intake",
    "recipient": "security",
    "intent": "escalation",
    "sensitivity": "restricted",
    "payload": {"detail": "summary 필드가 없음"},
}

for raw_message in [valid_message, invalid_edge, invalid_capability, invalid_schema]:
    result = graph.invoke({"raw_message": raw_message, "audit_log": []})
    print("=" * 80)
    print("Trace:", raw_message.get("trace_id"))
    print("Result:", result["delivery_result"])
    print("Errors:", result.get("validation_errors"))
    pprint(result["audit_log"])


## 정리

- 메시지 스키마 검증과 정책 검증은 서로 다른 단계입니다.
- 허용된 edge, 수신자 Capability, 데이터 민감도를 모두 확인해야 합니다.
- 차단된 메시지도 감사 로그에 남겨야 사고 분석이 가능합니다.
